# 🏥 Lab 0: Data Exploration & Preprocessing
**BINF 4002 – Machine Learning for Health**

---
## Learning Objectives
1. Load and inspect a real-world clinical dataset
2. Perform exploratory data analysis (EDA)
3. Identify and handle missing values correctly
4. Encode and scale features — and understand *why* each step matters
5. Split data properly to avoid data leakage
6. Understand what can go wrong at each step, using concrete clinical examples
7. Save cleaned data for all subsequent model labs

## Dataset
**Breast Cancer Wisconsin** (`sklearn.datasets`) — 569 samples, 30 numeric features
derived from cell-nucleus images; binary outcome: malignant (0) vs. benign (1).

> ⚠️ **Reproducibility note**: set a random seed everywhere. Results that cannot be
> reproduced cannot be trusted in clinical research.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import f_classif
import pickle, warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette("Set2")

raw = load_breast_cancer()
df  = pd.DataFrame(raw.data, columns=raw.feature_names)
df['target']    = raw.target           # 0 = malignant, 1 = benign
df['diagnosis'] = df['target'].map({0:'Malignant', 1:'Benign'})

print(f"Shape: {df.shape}")
print(f"Classes:\n{df['diagnosis'].value_counts()}")
df.head(3)


---
## Part 1 — First Look at the Data

Before running any model, answer: *What is our data? What is our problem? What are we predicting? For whom? With what features?*
These questions seem obvious, but skipping them is the root cause of many published ML failures in healthcare.

For now, take a moment to look up information about the source of this data (in the real world, you might need to ask colleagues about the data, or work with clinical or biology collaborators to understand how and why the data were generated).

**Add a text cell below summarizing some of the essential aspects of the data. What kinds of problems could this data help solve? In addition, describe a plan for how you might do exploratory data analysis on these data (e.g., to investigate the data to better understand it).**

Below, we do show some basic aspects of the data. Feel free to extend this with your own data anlaysis code!

In [ ]:
print("Feature summary:")
df.drop(columns=['target','diagnosis']).describe().T[['mean','std','min','max']].round(3)

### 🤔 Reflection 1.1 — Clinical Context

Before looking at the features more carefully, answer these questions:

1. These features are derived from cell-nucleus images. Who collects this data in a hospital?
   Would every hospital collect it in the same way? What problems might arise if a model
   trained at one hospital is deployed at another?

2. The dataset has 357 benign and 212 malignant cases (~63% benign).
   If you built a model that **always predicts "benign"**, what would its accuracy be?
   Is that a useful model? What does this tell you about using accuracy as your primary metric?

3. We have 569 samples and 30 features. Is this a big dataset for healthcare? Compare to
   what you might have in a clinical trial vs. an EHR system. How does dataset size affect
   what models are appropriate?


In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(11,4))

counts = df['diagnosis'].value_counts()
axes[0].bar(counts.index, counts.values,
            color=['#e74c3c','#2ecc71'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i,v in enumerate(counts.values):
    axes[0].text(i, v+5, str(v), ha='center', fontweight='bold')

# Naive-classifier baseline
naive_acc = counts.max() / counts.sum()
axes[0].set_xlabel(f'Naive accuracy (always predict majority): {naive_acc:.1%}')

# Feature: mean radius by class
for label, color, name in zip([0,1],['#e74c3c','#2ecc71'],['Malignant','Benign']):
    axes[1].hist(df[df['target']==label]['mean radius'], bins=25,
                 alpha=0.6, label=name, color=color, edgecolor='white')
axes[1].set_title('Mean Radius by Class')
axes[1].set_xlabel('Mean Radius (µm)')
axes[1].legend()

plt.tight_layout(); plt.show()


---
## Part 2 — Feature Distributions and Outliers

Real clinical data is messy. Features can have extreme outliers, skewed distributions, or
values that are impossible given the measurement context.


In [ ]:
# Distribution of all features — look for skew and outliers
feat_cols = raw.feature_names.tolist()
n_cols = 6
n_rows = int(np.ceil(len(feat_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows*2.2))

for ax, col in zip(axes.ravel(), feat_cols):
    for label, color in zip([0,1],['#e74c3c','#2ecc71']):
        ax.hist(df[df['target']==label][col], bins=20,
                alpha=0.5, color=color, edgecolor='none')
    ax.set_title(col, fontsize=7)
    ax.tick_params(labelsize=6)
    ax.set_yticks([])

for ax in axes.ravel()[len(feat_cols):]:
    ax.axis('off')

plt.suptitle('Feature Distributions by Class (Red=Malignant, Green=Benign)', y=1.01, fontsize=11)
plt.tight_layout(); plt.show()


### 🤔 Reflection 2.1 — Distributions and Clinical Meaning

1. Some features (e.g., "worst area") have very long right tails. Name two ways this could
   cause problems for a linear model. How would you check whether this is hurting your model?

2. Several features are clearly bimodal — they separate the classes well visually.
   Does a large visual separation guarantee a model will perform well? What else matters?

3. In a real EHR, you might find a feature like "cholesterol = 0" that isn't a true zero
   but a missing value coded as 0 by the data entry system. How would you detect this?
   What would you do about it?

4. If a feature's distribution differs between malignant and benign cases, is that feature
   necessarily *causal*? Give an example where a feature is predictive but not causal,
   and explain why the distinction matters for clinical decision making.


---
## Part 3 — Correlation and Multicollinearity

In the breast cancer dataset, radius, perimeter, and area are geometrically related:
perimeter ≈ 2πr, area ≈ πr². This means they carry almost the same information.

This is common in clinical data: BMI is derived from height and weight; eGFR is derived from
creatinine; many lab panels share underlying biological processes.

One way we can measure these relationships is to look at the [_correlation_](https://en.wikipedia.org/wiki/Correlation) between variables. Recall that the correlation between two random variables $X$ and $Y$ is given by:

$$\rho_{X, Y} = \frac{\text{Cov}(X, Y)}{\sqrt{\text{Var}(X)\text{Var}(Y)}}$$.

The code below measures and visualizes these correlations across all our features:

In [ ]:
# Correlation heatmap
corr = df[feat_cols].corr()

plt.figure(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            vmin=-1, vmax=1, annot=False, linewidths=0.2,
            xticklabels=True, yticklabels=True)
plt.title('Feature Correlation Matrix — Breast Cancer Dataset', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout(); plt.show()

# Count highly correlated pairs
high_corr = (corr.abs() > 0.85) & (corr != 1.0)
n_high = high_corr.values.sum() // 2
print(f"Feature pairs with |r| > 0.85: {n_high}")
print("\nTop 10 most correlated pairs:")
pairs = [(corr.index[i], corr.columns[j], corr.iloc[i,j])
         for i in range(len(corr)) for j in range(i)
         if abs(corr.iloc[i,j]) > 0.85]
pairs_df = pd.DataFrame(pairs, columns=['Feature 1','Feature 2','r']).sort_values('r', ascending=False, key=abs)
print(pairs_df.head(10).to_string(index=False))


### 🤔 Reflection 3.1 — Why Multicollinearity Matters

1. If `mean radius` and `mean perimeter` have r = 0.998, how might this effect your modeling pipeline? Could it hurt your model's *predictive* performance? Could it hurt your model's *interpretability*? Explain your answer.

2. Name a common clinical dataset where you'd expect severe multicollinearity among lab values.

3. Suppose you wanted to keep only a subset of features. How might a clinician's intuition help you decide which features to keep?

4. Suppose you wanted your features to all not be correlated. How could you accomplish this? *Hint: How could you transform your input features to remove correlation?* What problems could this solve, and what problems could this create/worsen? Try and implement a version of this below.

5. Here, we have only measured *correlation*, which is only a measure of linear dependence. We have seen a different, more complete metric for dependency in our lectures on probability and information theory. What metric is that? Could you try to visualize those relationships as well? Why or why not? What would that tell you? What would happen if you tried to control for that metric in the same way you could correlation?

6. This examination only looks at _binary_ relationships (e.g., how is $X$ related to $Y$), and ignores higher-order relationships (e.g., how is $(X, Y)$ related to $Z$). Is it possible for there to be a higher order relationship without a binary relationship? Can you think of any clinical settings where such higher-order relationships may be present? Why do we often ignore these higher-order relationships in practice? Why does or does this not cause problems?

---
## Part 4 — Simulating and Handling Missing Data

The breast cancer dataset has no missing values, but real clinical data almost always does.
We simulate realistic missingness to practice correct handling.

Vocabulary:
- MCAR = Missing Completely At Random
- MAR = Missing At Random
- MNAR = Missing Not At Random


In [ ]:
# Simulate missing data under two mechanisms
rng = np.random.default_rng(42)
df_missing = df[feat_cols + ['target']].copy()

# MCAR: Missing Completely At Random — random 5% dropout in 'mean smoothness'
idx_mcar = rng.choice(len(df_missing), size=int(0.05*len(df_missing)), replace=False)
df_missing.loc[idx_mcar, 'mean smoothness'] = np.nan

# MAR: Missing At Random — malignant cases 3x more likely to have missing 'mean concavity'
mal_idx = df_missing[df_missing['target']==0].index
miss_mal = rng.choice(mal_idx, size=int(0.20*len(mal_idx)), replace=False)
ben_idx  = df_missing[df_missing['target']==1].index
miss_ben = rng.choice(ben_idx,  size=int(0.07*len(ben_idx)),  replace=False)
df_missing.loc[np.concatenate([miss_mal, miss_ben]), 'mean concavity'] = np.nan

print("Missing values introduced:")
print(df_missing.isnull().sum()[df_missing.isnull().sum()>0])

# Visualize: is the missingness of 'mean concavity' related to class?
fig, ax = plt.subplots(figsize=(7,4))
for label, color, name in zip([0,1],['#e74c3c','#2ecc71'],['Malignant','Benign']):
    subset = df_missing[df_missing['target']==label]
    miss_rate = subset['mean concavity'].isna().mean()
    ax.bar(name, miss_rate*100, color=color, edgecolor='white')
    ax.text(name, miss_rate*100+0.5, f'{miss_rate:.1%}', ha='center', fontweight='bold')
ax.set_ylabel('% Missing'); ax.set_title("Missingness Rate in 'mean concavity' by Class\n(MAR: missingness depends on outcome)")
plt.tight_layout(); plt.show()


### 🤔 Reflection 4.1 — Missing Data Mechanisms

1. **MCAR vs MAR vs MNAR**: We simulated MCAR (mean smoothness) and MAR (mean concavity).
   Describe what "Missing Not at Random" (MNAR) would look like in a clinical setting.
   Give a concrete example involving blood pressure measurements.

2. For the `mean concavity` feature (MAR), malignant cases are 3× more likely to have
   missing values. If you simply drop rows with missing values, what bias does this
   introduce into your training set? Which class becomes under-represented?

3. We're about to use mean imputation. Under what distributional assumption is
   mean imputation "correct"? Does that assumption hold here?

4. Why is it critical to fit the imputer **only on training data** and then apply it to
   validation/test data? What specific number would "leak" if you fit on all data?


In [ ]:
# Compare imputation strategies
from sklearn.impute import SimpleImputer, KNNImputer

strategies = {
    'Mean imputation': SimpleImputer(strategy='mean'),
    'Median imputation': SimpleImputer(strategy='median'),
    'KNN imputation (k=5)': KNNImputer(n_neighbors=5),
}

# Use only training data to fit imputers (correct approach)
# We'll do a quick train/val split of df_missing for demonstration
from sklearn.model_selection import train_test_split
X_m = df_missing[feat_cols].values
y_m = df_missing['target'].values
Xm_tr, Xm_val, _, _ = train_test_split(X_m, y_m, test_size=0.3, random_state=42)

col_idx_concavity = feat_cols.index('mean concavity')
print(f"'mean concavity' — column index: {col_idx_concavity}")
print(f"True train mean (non-missing): {np.nanmean(Xm_tr[:, col_idx_concavity]):.4f}")
print()

for name, imp in strategies.items():
    imp.fit(Xm_tr)                     # ← fit on train only
    Xm_tr_imp = imp.transform(Xm_tr)
    Xm_val_imp = imp.transform(Xm_val) # ← apply to val
    remaining_miss = np.isnan(Xm_val_imp).sum()
    imp_val = np.nanmean(Xm_val_imp[:, col_idx_concavity])
    print(f"{name}: imputed val mean = {imp_val:.4f},  remaining missing = {remaining_miss}")

print("\n⚠️  Wrong approach (data leakage): fitting imputer on ALL data before splitting")
leaky_imp = SimpleImputer(strategy='mean')
leaky_imp.fit(X_m)   # fits on val/test too!
print(f"Leaky imputed mean = {leaky_imp.statistics_[col_idx_concavity]:.4f}  (uses test info)")


---
## Part 5 — Feature Scaling

Many algorithms assume features are on comparable scales. Without scaling, a feature
measured in thousands (e.g., area in µm²) will dominate one measured in fractions (e.g., smoothness).


In [ ]:
# Show scale differences across features
scales = df[feat_cols].std().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(scales.index, scales.values, color='#9b59b6', edgecolor='white')
ax.set_ylabel('Standard Deviation (raw scale)')
ax.set_title('Feature Standard Deviations — Raw Data\n(Huge range: algorithms that use distances or gradients will be dominated by large-scale features)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

print(f"Largest std: {scales.index[0]}  = {scales.iloc[0]:.1f}")
print(f"Smallest std: {scales.index[-1]} = {scales.iloc[-1]:.4f}")
print(f"Ratio: {scales.iloc[0]/scales.iloc[-1]:.0f}x difference in scale")


### 🤔 Reflection 5.1 — When Does Scaling Matter?

Below, we ask about some different types of models. These will also be explored more deeply in the model-specific notebooks.

1. **Logistic regression**: Would the *predictions* of a logistic regression change if you
   forgot to scale? Would the *coefficients* change? Explain why or why not.
   (Hint: think about what happens to the weights when features have very different scales.)

2. **k-Nearest Neighbors**: Explain concretely why scaling is absolutely critical for KNN.
   Give an example using two features — one in millimeters and one in kilograms — and
   describe what would happen to the "nearest neighbor" calculation without scaling.

3. **Decision trees**: Why are decision trees invariant to monotonic transformations of
   features (including scaling)?

4. **Neural Networks**: Suppose we are training a neural network with stochastic gradient descent. What impact does feature scale have on the gradients computed for a loss in this setting? _Hint: Try examining the gradient of the weights of a linear layer with respect to a 1D output._

5. **Train-only fitting**: We fit the scaler on training data and apply it to test data.
   What specific numbers would be "contaminated" if we fit on all data? Write out
   the StandardScaler formula and identify which terms would be wrong.


In [ ]:
# Train / Val / Test split first (stratified)
X_full = df[feat_cols].values
y_full = df['target'].values

X_train_f, X_tmp, y_train_f, y_tmp = train_test_split(
    X_full, y_full, test_size=0.40, random_state=42, stratify=y_full)
X_val_f, X_test_f, y_val_f, y_test_f = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp)

print(f"Full splits — Train: {X_train_f.shape[0]}  Val: {X_val_f.shape[0]}  Test: {X_test_f.shape[0]}")

# Fit scaler on training data only
scaler_full = StandardScaler()
X_train_fs = scaler_full.fit_transform(X_train_f)
X_val_fs   = scaler_full.transform(X_val_f)
X_test_fs  = scaler_full.transform(X_test_f)

# Show before / after for worst area (largest range)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
col_wa = feat_cols.index('worst area')
for ax, data, title, color in zip(
    axes,
    [X_train_f[:,col_wa], X_train_fs[:,col_wa]],
    ['Before Scaling: "worst area"', 'After StandardScaler: "worst area"'],
    ['#e67e22','#27ae60']
):
    ax.hist(data, bins=30, color=color, edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('Value')
plt.tight_layout(); plt.show()
print(f"Before: mean={X_train_f[:,col_wa].mean():.1f}, std={X_train_f[:,col_wa].std():.1f}")
print(f"After:  mean={X_train_fs[:,col_wa].mean():.4f}, std={X_train_fs[:,col_wa].std():.4f}")


---
## Part 6 — Feature Selection and Creating a Realistic "Hard" Dataset

The full dataset with all 30 features gives AUC > 0.99 on real models — not realistic for most clinical problems. We'll create a restricted dataset by:
1. Removing the most informative features (simulating limited data collection)
2. Sub-sampling the training set (simulating small study sizes)

This creates a more realistic setting where model choice and preprocessing decisions actually matter. In practice, of course, you wouldn't do this, but for these notebooks, it will help make things easier.


In [ ]:
from sklearn.feature_selection import f_classif

# Rank features by univariate F-score on the full training set
f_scores, _ = f_classif(X_train_fs, y_train_f)
f_series = pd.Series(f_scores, index=feat_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(13, 4))
colors = ['#e74c3c' if i < 22 else '#95a5a6' for i in range(len(f_series))]
ax.bar(f_series.index, f_series.values, color=colors, edgecolor='white')
ax.set_ylabel('F-score')
ax.set_title('Feature Discriminative Power (F-score)\n'
             'Red = top 22 most informative features — EXCLUDED in "hard" dataset\n'
             'Gray = bottom 8 — KEPT (individually weak, more realistic challenge)')
ax.tick_params(axis='x', rotation=45, labelsize=8)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#e74c3c', label='Excluded (too discriminative)'),
                   Patch(color='#95a5a6', label='Kept for modeling labs')], loc='upper right')
plt.tight_layout(); plt.show()

print(f"Bottom 8 features (kept for modeling labs):")
print(f_series.index[22:].tolist())


In [ ]:
# Build "hard" splits:
#   - Keep only the BOTTOM 8 features by F-score (least individually informative)
#   - Sub-sample training to 100 examples (realistic small clinical study)
# This gives AUC ~0.83–0.86 — realistic for a constrained clinical dataset.

hard_features = f_series.index[22:].tolist()   # bottom 8 by F-score
hard_col_idx  = [feat_cols.index(c) for c in hard_features]

X_train_h = X_train_fs[:, hard_col_idx]
X_val_h   = X_val_fs[:, hard_col_idx]
X_test_h  = X_test_fs[:, hard_col_idx]

# Sub-sample training to 100 examples
rng2 = np.random.default_rng(42)
n_hard_train = 100
hard_idx = rng2.choice(len(X_train_h), size=n_hard_train, replace=False)
X_train_hard = X_train_h[hard_idx]
y_train_hard = y_train_f[hard_idx]

X_val_hard  = X_val_h
y_val_hard  = y_val_f
X_test_hard = X_test_h
y_test_hard = y_test_f

print(f"Hard training set: {X_train_hard.shape}")
print(f"Hard val set:      {X_val_hard.shape}")
print(f"Hard test set:     {X_test_hard.shape}")
print(f"Training prevalence: {y_train_hard.mean():.2%} benign")
print(f"Features kept: {hard_features}")

# Quick sanity: what AUC does a simple model achieve?
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
lr_sanity = LogisticRegression(C=1.0, max_iter=500, random_state=42)
lr_sanity.fit(X_train_hard, y_train_hard)
auc_sanity = roc_auc_score(y_val_hard, lr_sanity.predict_proba(X_val_hard)[:,1])
print(f"\nLogistic regression AUC on hard val set: {auc_sanity:.3f}  (target: 0.80 – 0.90)")


### 🤔 Reflection 6.1 — What Makes a Problem Hard?

1. We removed the top-10 most informative features. In a real clinical study, why might
   you not have access to the most informative biomarkers? Give two examples
   (one from cost, one from timing).

2. We sub-sampled the training set to 100 examples. What are the statistical consequences
   of a small training set? How does this interact with model complexity (number of parameters)?

3. With 100 training samples and 8 features, are you at risk of overfitting even with
   a simple logistic regression? How many parameters does logistic regression have here?

4. In practice, should you always use all available features? Describe a scenario where
   adding more features *hurts* generalization performance.


---
## Part 7 — Save Processed Data

All subsequent model labs will load this file.


In [ ]:
processed = {
    # Full splits (all features, full training size)
    'X_train': X_train_fs, 'y_train': y_train_f,
    'X_val':   X_val_fs,   'y_val':   y_val_f,
    'X_test':  X_test_fs,  'y_test':  y_test_f,
    'feature_names': feat_cols,
    # Hard splits (restricted features, sub-sampled training)
    'X_train_hard': X_train_hard, 'y_train_hard': y_train_hard,
    'X_val_hard':   X_val_hard,   'y_val_hard':   y_val_hard,
    'X_test_hard':  X_test_hard,  'y_test_hard':  y_test_hard,
    'feature_names_hard': hard_features,
    # Metadata
    'scaler': scaler_full,
    'class_names': raw.target_names.tolist(),
    'hard_col_idx': hard_col_idx,
}

with open('processed_data.pkl', 'wb') as f:
    pickle.dump(processed, f)

print("✅ Saved processed_data.pkl")
print(f"   Keys: {list(processed.keys())}")


---
## 🧠 Final Reflection — The Preprocessing Pipeline as a Whole

Answer all of these before moving to the model labs:

1. **Order of operations**: Why must you (a) split first, (b) then fit imputer/scaler on
   train, (c) then transform val/test? Draw the pipeline as a diagram and mark where
   data leakage could occur at each step.

2. **Clinical deployment**: Your preprocessing pipeline includes a scaler fitted on your
   training data. When you deploy this model at a new hospital six months later, what
   do you need to deploy alongside the model weights? What happens if you forget?

3. **Distribution shift**: The new hospital's patient population has different demographics
   than your training set. Which steps of your preprocessing pipeline are most sensitive
   to this shift? What would you monitor in production?

4. **Reproducibility crisis**: A published paper reports 0.98 AUC on a breast cancer dataset.
   Name three preprocessing mistakes (beyond the ones covered in this lab) that could
   cause this number to be inflated. How would you audit the paper?
